# Fine-tune QLoRA — Qwen2.5-3B (nhanh + lưu Google Drive)
*Chuẩn bị:* đặt file `dataset.jsonl` vào Google Drive thư mục `MyDrive/qwen-finetune/` (hoặc upload trực tiếp vào Colab).
*Cấu hình nhanh:* 1 epoch. Muốn 2 epochs thì đổi `num_train_epochs`.

In [ ]:
# 1. Cài Unsloth
!pip install -q unsloth

In [ ]:
# 2. Gắn Google Drive (lưu checkpoint/GGUF)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/qwen-finetune"
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Drive OK:", DRIVE_DIR)

In [ ]:
# 3. Tải model 4-bit + gắn LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=512,          # giảm xuống cho NHANH; nâng 1024 nếu cần tóm tắt dài
    dtype=None, load_in_4bit=True,
    device_map="auto",
)
model = FastLanguageModel.get_peft_model(
    model, r=8, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

In [ ]:
# 4. Nạp dataset (ưu tiên từ Drive, nếu không có thì dùng file upload)
import json, os, random
from datasets import Dataset

candidates = [
    f"{DRIVE_DIR}/dataset.jsonl",
    "dataset.jsonl",   # file upload trực tiếp lên Colab
]
path = next((p for p in candidates if os.path.exists(p)), None)
assert path, "Khong tim thay dataset.jsonl (copy len Drive hoac upload vao Colab)"

samples = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
random.Random(42).shuffle(samples)
split = int(len(samples) * 0.9)
train_data, test_data = samples[:split], samples[split:]
print(f"Train: {len(train_data)}, Test: {len(test_data)}, nguon: {path}")

def build_prompt(ex):
    user = f"{ex['instruction']}\n{ex['input']}" if ex.get("input") else ex["instruction"]
    return {"prompt": user, "completion": ex["output"]}

train_ds = Dataset.from_list([build_prompt(x) for x in train_data])
def format_row(r):
    return tokenizer.apply_chat_template(
        [{"role":"user","content":r["prompt"]},
         {"role":"assistant","content":r["completion"]}],
        tokenize=False, add_generation_prompt=False)
train_ds = train_ds.map(lambda r: {"text": format_row(r)})

In [ ]:
# 5. Huấn luyện — 1 epoch, checkpoint tự lưu lên DRIVE mỗi 200 bước
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

OUT = f"{DRIVE_DIR}/outputs"
os.makedirs(OUT, exist_ok=True)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds,
    dataset_text_field="text", max_seq_length=512,
    args=TrainingArguments(
        output_dir=OUT,                     # checkpoint nằm trên DRIVE
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=1,                 # 1 epoch nhanh; muốn chất lượng hơn -> 2
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=10,
        save_strategy="steps",
        save_steps=200,                     # cứ 200 bước lưu checkpoint lên Drive
        save_total_limit=3,
        optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=42,
    ),
)

# Nếu lần trước bị ngắt, tiếp tục từ checkpoint mới nhất trên Drive
trainer.train(resume_from_checkpoint=True)

In [ ]:
# 6. Merge + export GGUF Q4_K_M -> LƯU LÊN DRIVE
model.save_pretrained_merged(f"{DRIVE_DIR}/merged_model", tokenizer)
model.save_pretrained_gguf(f"{DRIVE_DIR}/gguf_output", tokenizer, quantization_method="q4_k_m")
import glob
print("GGUF:", glob.glob(f"{DRIVE_DIR}/gguf_output/*.gguf"))
# Xong: file GGUF nằm trong Drive -> tải về máy từ Drive, không sợ mất

In [ ]:
# 7. (Tuỳ chọn) Đánh giá nhanh trên tập test
from transformers import TextStreamer
FastLanguageModel.for_inference(model)
text = tokenizer.apply_chat_template(
    [{"role":"user","content":test_data[0]["instruction"]}],
    tokenize=False, add_generation_prompt=True)
out = model.generate(**tokenizer([text], return_tensors="pt").to("cuda"), max_new_tokens=256)
print(tokenizer.decode(out[0], skip_special_tokens=True))